---
execute:
    echo: false
---

# Elite Dangerous Database Reader
> Read Elite Dangerous database files

In [ ]:
#| default_exp eddb.readers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, time, logging, json, gzip, os
import edcompanion.core

from pathlib import Path
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging


In [ ]:
init_console_logging(__name__)

2025-12-19T11:22:58+0100 INFO	2632	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
syslog.info(f"Loading module {__name__}")

In [ ]:
eddb_conf = configuration['EDDB']

In [ ]:
#| export
def dbfilereader(filename):
    """
        Opens 'filename' as generator for eddb style objects
    """

    chunksize = 64 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                for line in chunk:
                    if len(line) < 5:
                        continue

                    yield json.loads(line.rstrip(',\n\r '))

            else:
                break




In [ ]:
systems1 = os.path.join(eddb_conf['localdumps'], eddb_conf['systems_1day'])
print(systems1)

D:\data\eddb\systems_1day.json.gz


In [ ]:
i = 0
for item in dbfilereader(systems1):
    print(item)
    i+=1
    if i > 10:
        break

{'id64': 2587799, 'name': 'CD-26 1339', 'mainStar': 'O (Blue-White) Star', 'coords': {'x': 437.21875, 'y': -925.15625, 'z': -513.78125}, 'updateTime': '2025-12-18 19:58:47+00'}
{'id64': 2850943, 'name': 'Gludgoo AA-A h0', 'mainStar': 'Wolf-Rayet O Star', 'coords': {'x': 5791.5, 'y': 67.59375, 'z': -4838.71875}, 'updateTime': '2025-12-18 18:57:05+00'}
{'id64': 11239631, 'name': 'Prielee AA-A h1', 'mainStar': 'B (Blue-White super giant) Star', 'coords': {'x': 5160.5625, 'y': 218.78125, 'z': 8009.03125}, 'updateTime': '2025-12-18 10:34:58+00'}
{'id64': 12710694, 'name': 'Trigna AA-A g0', 'mainStar': 'O (Blue-White) Star', 'coords': {'x': -18977.3125, 'y': -1111.21875, 'z': 40114.84375}, 'updateTime': '2025-12-18 12:24:37+00'}
{'id64': 12974886, 'name': 'Trigna ZE-A g0', 'mainStar': 'B (Blue-White) Star', 'coords': {'x': -18216.46875, 'y': -137.125, 'z': 40047.46875}, 'updateTime': '2025-12-18 12:46:40+00'}
{'id64': 19365015, 'name': 'BD-12 1172', 'mainStar': 'O (Blue-White) Star', 'coords

In [ ]:
#| export

async def dbfile_process_async(filename, process_chunk):
    """Opens file and calls process_chunk to process batches of items"""

    chunksize = 16 * 1024 * 1024

    with gzip.open(filename, 'rt') as jsonfile:

        while True:
            chunk = jsonfile.readlines(chunksize)
            if chunk:
                data = []
                for line in chunk:
                    if len(line) < 5:
                        continue

                    item = json.loads(line.rstrip(',\n\r '))
                    data.append(item)

                await process_chunk(data)

            else:
                break



In [ ]:
#| hidey
import nbdev; nbdev.nbdev_export()